# 06 - ML Observability with Model Monitors

This notebook sets up automated model monitoring for drift detection and performance tracking.

**Key Concepts**:
- **Model Monitor**: A Snowflake object that continuously tracks model health
- **Drift Detection**: PSI (Population Stability Index), KL Divergence, Difference of Means
- **Performance Tracking**: F1, Precision, Recall, AUC over time
- **Volume Monitoring**: Track prediction volume trends
- **Automated Refresh**: Configurable refresh interval

**Prerequisites**: Run `02_model_training_xgboost.ipynb` and `05_batch_inference.ipynb` first.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE DATABASE MLOPS_DEMO_DB").collect()
print(f"Connected: {session.get_current_user()}")

## Monitoring Concepts

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **PSI** | Distribution shift in feature values | Detects when input data changes significantly |
| **KL Divergence** | Information loss between distributions | More sensitive to tail changes than PSI |
| **Difference of Means** | Shift in central tendency | Simple, interpretable drift signal |
| **F1 / Precision / Recall** | Classification performance | Tracks if the model is still accurate |
| **AUC** | Ranking quality | Measures discrimination ability over time |

In [ ]:
# Create a prediction log table with actuals (needed for performance monitoring)
session.sql("""
CREATE OR REPLACE TABLE MLOPS_DEMO_DB.MONITORING.CHURN_PREDICTION_LOG AS
SELECT 
    p.CUSTOMER_ID,
    p.output_feature_0 AS PREDICTED_CHURN,
    c.EXITED AS ACTUAL_CHURN,
    p.PREDICTION_TIMESTAMP,
    c.CREDIT_SCORE,
    c.AGE,
    c.TENURE,
    c.BALANCE,
    c.NUM_OF_PRODUCTS,
    c.HAS_CR_CARD,
    c.IS_ACTIVE_MEMBER,
    c.ESTIMATED_SALARY
FROM MLOPS_DEMO_DB.OUTPUT.CHURN_PREDICTIONS p
JOIN MLOPS_DEMO_DB.RAW.CUSTOMERS c ON p.CUSTOMER_ID = c.CUSTOMER_ID
""").collect()

count = session.table("MLOPS_DEMO_DB.MONITORING.CHURN_PREDICTION_LOG").count()
print(f"Prediction log created: {count} rows")

In [ ]:
# Verify the prediction log structure
session.table("MLOPS_DEMO_DB.MONITORING.CHURN_PREDICTION_LOG").show(5)

In [ ]:
# Create Model Monitor
# This sets up automated drift and performance tracking
session.sql("""
CREATE OR REPLACE MODEL MONITOR MLOPS_DEMO_DB.MONITORING.CHURN_MONITOR
WITH
    MODEL = MLOPS_DEMO_DB.PIPELINES.CHURN_PREDICTION_MODEL
    VERSION = v1
    FUNCTION = predict
    SOURCE = MLOPS_DEMO_DB.MONITORING.CHURN_PREDICTION_LOG
    TIMESTAMP_COLUMN = PREDICTION_TIMESTAMP
    PREDICTION_COLUMN = PREDICTED_CHURN
    ACTUAL_COLUMN = ACTUAL_CHURN
    ID_COLUMNS = (CUSTOMER_ID)
    REFRESH_INTERVAL = '1 HOUR'
""").collect()
print("Model Monitor created: CHURN_MONITOR")
print("Refresh interval: 1 hour")

In [ ]:
# Verify the monitor exists
session.sql("""
SHOW MODEL MONITORS IN SCHEMA MLOPS_DEMO_DB.MONITORING
""").show()

In [ ]:
# Query drift metrics (PSI)
# Note: Metrics become available after the first refresh cycle
try:
    drift = session.sql("""
    SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
        'MLOPS_DEMO_DB.MONITORING.CHURN_MONITOR',
        METRIC_TYPE => 'PSI'
    ))
    """).to_pandas()
    print("Drift Metrics (PSI):")
    print(drift.to_string(index=False))
except Exception as e:
    print(f"Drift metrics not yet available (first refresh pending): {e}")
    print("Metrics will populate after the first refresh interval.")

In [ ]:
# Query performance metrics (F1 Score)
try:
    perf = session.sql("""
    SELECT * FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
        'MLOPS_DEMO_DB.MONITORING.CHURN_MONITOR',
        METRIC_TYPE => 'F1_SCORE'
    ))
    """).to_pandas()
    print("Performance Metrics (F1 Score):")
    print(perf.to_string(index=False))
except Exception as e:
    print(f"Performance metrics not yet available: {e}")
    print("Metrics will populate after the first refresh interval.")

In [ ]:
# You can also manually trigger a refresh
try:
    session.sql("""
    ALTER MODEL MONITOR MLOPS_DEMO_DB.MONITORING.CHURN_MONITOR REFRESH
    """).collect()
    print("Manual refresh triggered. Metrics will be available shortly.")
except Exception as e:
    print(f"Note: {e}")

## What to Show in SnowSight

Navigate to **AI/ML > Model Registry > CHURN_PREDICTION_MODEL > v1 > Monitors**:

- **Drift Dashboard**: Visual charts showing feature distribution shifts over time
- **Performance Dashboard**: F1, precision, recall trends
- **Volume Chart**: Prediction count over time
- **Alert Configuration**: Set thresholds for automated alerts

**Key message**: Model monitoring is a native Snowflake capability -- no external tools like Evidently, WhyLabs, or custom dashboards needed. One SQL statement sets up continuous monitoring.

---

**Next**: `07_explainability_lineage.ipynb` - SHAP explanations and ML lineage